In [ ]:
import json
import random
from collections import defaultdict

PHASE3_DIR = "/content/drive/MyDrive/PACL_Project/phase3_outputs"


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]


train_records = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
print("train:", len(train_records), "| شاعرانِ یکتا:", len(set(r["poet"] for r in train_records)))

poet_to_indices = defaultdict(list)
poet_to_school = {}
for idx, r in enumerate(train_records):
    poet_to_indices[r["poet"]].append(idx)
    poet_to_school[r["poet"]] = r.get("school")

school_to_poets = defaultdict(list)
for poet, school in poet_to_school.items():
    school_to_poets[school].append(poet)

print("\nتوزیعِ شاعران بر اساسِ مکتب:")
for school, poets in sorted(school_to_poets.items(), key=lambda x: -len(x[1])):
    print(f"  {school}: {len(poets)} شاعر")


class SchoolGroupedPKBatchSampler:

    def __init__(
        self,
        poet_to_indices,
        school_to_poets,
        poet_to_school,
        primary_poets_per_batch=3,
        secondary_poets_per_batch=1,
        poems_per_poet=2,
        seed=42,
    ):
        self.poet_to_indices = poet_to_indices
        self.school_to_poets = school_to_poets
        self.poet_to_school = poet_to_school
        self.primary_poets_per_batch = primary_poets_per_batch
        self.secondary_poets_per_batch = secondary_poets_per_batch
        self.poems_per_poet = poems_per_poet
        self.seed = seed
        self.all_poets = list(poet_to_indices.keys())

    def __iter__(self):
        rng = random.Random(self.seed)
        poet_order = self.all_poets[:]
        rng.shuffle(poet_order)

        primary_groups = [
            poet_order[i : i + self.primary_poets_per_batch]
            for i in range(0, len(poet_order), self.primary_poets_per_batch)
        ]

        for primary_poets in primary_groups:
            batch_poets = list(primary_poets)
            primary_school = self.poet_to_school.get(primary_poets[0])

            other_school_poets = [
                p
                for s, poets in self.school_to_poets.items()
                if s != primary_school
                for p in poets
                if p not in batch_poets
            ]
            rng.shuffle(other_school_poets)
            batch_poets.extend(other_school_poets[: self.secondary_poets_per_batch])

            batch_indices = []
            for poet in batch_poets:
                indices = self.poet_to_indices[poet]
                if len(indices) >= self.poems_per_poet:
                    chosen = rng.sample(indices, self.poems_per_poet)
                else:
                    chosen = [rng.choice(indices) for _ in range(self.poems_per_poet)]
                batch_indices.extend(chosen)

            yield batch_indices

    def __len__(self):
        return (len(self.all_poets) + self.primary_poets_per_batch - 1) // self.primary_poets_per_batch


sampler = SchoolGroupedPKBatchSampler(
    poet_to_indices,
    school_to_poets,
    poet_to_school,
    primary_poets_per_batch=3,
    secondary_poets_per_batch=1,
    poems_per_poet=2,
    seed=42,
)

covered_poets = set()
batch_sizes = []
for batch_indices in sampler:
    covered_poets |= set(train_records[i]["poet"] for i in batch_indices)
    batch_sizes.append(len(batch_indices))

all_poets = set(poet_to_indices.keys())
missing = all_poets - covered_poets

print("\n--- نتیجه‌ی تستِ Coverage (روی phase3_outputs) ---")
print("تعداد کل شاعران:", len(all_poets), "(باید ۱۵۰ باشد)")
print("شاعرانِ پوشش‌داده‌شده:", len(covered_poets))
print("شاعرانِ جامانده:", missing)
print("تعداد بچ‌ها:", len(batch_sizes))
print("میانگینِ اندازه‌ی بچ:", sum(batch_sizes) / len(batch_sizes))

assert len(all_poets) == 150, f"❌ انتظارِ ۱۵۰ شاعر داشتیم، {len(all_poets)} یافت شد!"
assert missing == set(), f"❌ {len(missing)} شاعر پوشش داده نشدند: {missing}"
print("\n✅ همه‌ی ۱۵۰ شاعر دقیقاً پوشش داده شدند.")

In [ ]:
import torch
import torch.nn.functional as F

def compute_d_struct(meta_v, meta_s, meta_q, alpha=1/3, beta=1/3, gamma=1/3):
    def match_indicator(x):
        eq = x.unsqueeze(0) == x.unsqueeze(1)
        unknown = (x.unsqueeze(0) == -1) | (x.unsqueeze(1) == -1)
        return (eq & ~unknown).float()

    Iv = match_indicator(meta_v)
    Is = match_indicator(meta_s)
    Iq = match_indicator(meta_q)
    return 1 - (alpha * Iv + beta * Is + gamma * Iq)

def compute_w(d_struct, tau_w=0.5):
    return torch.exp((1 - d_struct) / tau_w)

def ld_supcon_loss(z, labels, meta_v, meta_s, meta_q,
                    tau=0.07, tau_w=0.5, alpha=1/3, beta=1/3, gamma=1/3, eps=1e-8):
    B = z.shape[0]
    device = z.device

    z = F.normalize(z, dim=1)
    sim = torch.matmul(z, z.t()) / tau

    d_struct = compute_d_struct(meta_v, meta_s, meta_q, alpha, beta, gamma)
    w = compute_w(d_struct, tau_w)

    labels_eq = labels.unsqueeze(0) == labels.unsqueeze(1)
    self_mask = torch.eye(B, dtype=torch.bool, device=device)
    pos_mask = labels_eq & ~self_mask
    neg_mask = ~labels_eq

    exp_sim = torch.exp(sim)
    neg_sum = (w * exp_sim * neg_mask.float()).sum(dim=1)

    log_denom = torch.log(exp_sim + neg_sum.unsqueeze(1) + eps)
    log_prob = sim - log_denom

    pos_count = pos_mask.sum(dim=1)
    loss_per_anchor = -(log_prob * pos_mask.float()).sum(dim=1) / (pos_count.float() + eps)

    valid = pos_count > 0
    return loss_per_anchor[valid].mean()

In [ ]:
import json, torch
from collections import defaultdict

PHASE3_DIR = "/content/drive/MyDrive/PACL_Project/phase3_outputs"

records = []
with open(f"{PHASE3_DIR}/train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))
print("تعداد کل رکوردهایِ train:", len(records))

poet_to_indices = defaultdict(list)
for idx, r in enumerate(records):
    poet_to_indices[r["poet"]].append(idx)

sample_poets = list(poet_to_indices.keys())[:4]
batch_indices = []
for p in sample_poets:
    batch_indices.extend(poet_to_indices[p][:2])
batch_records = [records[i] for i in batch_indices]
B = len(batch_records)
print("اندازه‌ی batch:", B)
print("شاعرانِ این batch:", [r["poet"] for r in batch_records])
print("مکاتبِ این batch:", [r.get("school") for r in batch_records])

poet_to_id   = {p: i for i, p in enumerate(sorted(set(r["poet"] for r in records)))}
meter_to_id  = {m: i for i, m in enumerate(sorted(set(r.get("meter")  for r in records if r.get("meter"))))}
school_to_id = {s: i for i, s in enumerate(sorted(set(r.get("school") for r in records if r.get("school"))))}
form_to_id   = {q: i for i, q in enumerate(sorted(set(r.get("form")   for r in records if r.get("form"))))}

labels = torch.tensor([poet_to_id[r["poet"]] for r in batch_records], dtype=torch.long)
meta_v = torch.tensor([meter_to_id.get(r.get("meter"), -1)  for r in batch_records], dtype=torch.long)
meta_s = torch.tensor([school_to_id.get(r.get("school"), -1) for r in batch_records], dtype=torch.long)
meta_q = torch.tensor([form_to_id.get(r.get("form"), -1)    for r in batch_records], dtype=torch.long)

torch.manual_seed(42)
dummy_z = torch.randn(B, 128)

loss = ld_supcon_loss(dummy_z, labels, meta_v, meta_s, meta_q)
print("LD-SupCon loss (روی embeddingِ تصادفی):", loss.item())
assert loss.item() > 0, "لاس باید مثبت باشد"
print("✅ تابع بدون خطا اجرا شد و مقدارِ لاس معتبر برگرداند.")

In [ ]:
import json

PHASE3_DIR = "/content/drive/MyDrive/PACL_Project/phase3_outputs"
with open(f"{PHASE3_DIR}/train.jsonl", "r", encoding="utf-8") as f:
    sample = json.loads(f.readline())

print(json.dumps(sample, ensure_ascii=False, indent=2))

In [ ]:
import json, math, torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5EncoderModel, ByT5Tokenizer
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

PHASE3_DIR = "/content/drive/MyDrive/PACL_Project/phase3_outputs"
MODEL_NAME = "google/byt5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def clean(v):
    if v is None:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    return v

def load_records(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                recs.append(json.loads(line))
    return recs

records = load_records(f"{PHASE3_DIR}/train.jsonl")
print("تعداد کل رکوردهایِ train:", len(records))

def build_vocab(field):
    vals = {clean(r.get(field)) for r in records}
    vals.discard(None)
    return {v: i for i, v in enumerate(sorted(vals))}

poet_to_id   = build_vocab("poet")
rhythm_to_id = build_vocab("rhythm")
school_to_id = build_vocab("school")
format_to_id = build_vocab("format")

assert len(poet_to_id) == 150, f"باید ۱۵۰ شاعر باشد، شد {len(poet_to_id)}"
print("تعداد شاعران:", len(poet_to_id), "| تعداد وزن‌های یکتا:", len(rhythm_to_id),
      "| مکاتب:", len(school_to_id), "| قالب‌ها:", len(format_to_id))

class ByT5PACLEncoder(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-8)
        z = F.normalize(pooled, dim=1)
        return z

tokenizer = ByT5Tokenizer.from_pretrained(MODEL_NAME)
model = ByT5PACLEncoder().to(DEVICE)

LR = 3e-5
TARGET_BATCH_SIZE = 256
MICRO_BATCH_SIZE = 8
ACCUM_STEPS = TARGET_BATCH_SIZE // MICRO_BATCH_SIZE

SMOKE_TEST_MAX_OPT_STEPS = 5
STEPS_PER_EPOCH = len(sampler)
TOTAL_OPT_STEPS = math.ceil(STEPS_PER_EPOCH / ACCUM_STEPS)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_OPT_STEPS)

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available() and not USE_BF16)

print(f"ACCUM_STEPS={ACCUM_STEPS} → batch مؤثر={MICRO_BATCH_SIZE*ACCUM_STEPS} | AMP dtype={AMP_DTYPE}")
print(f"کلِ opt-stepهای یک epoch کامل: {TOTAL_OPT_STEPS} | اسموک‌تست فقط {SMOKE_TEST_MAX_OPT_STEPS} opt-step اجرا می‌کند")

def make_tensors(batch_records):
    labels = torch.tensor([poet_to_id[r["poet"]] for r in batch_records], dtype=torch.long)
    meta_v = torch.tensor([rhythm_to_id.get(clean(r.get("rhythm")), -1) for r in batch_records], dtype=torch.long)
    meta_s = torch.tensor([school_to_id.get(clean(r.get("school")), -1) for r in batch_records], dtype=torch.long)
    meta_q = torch.tensor([format_to_id.get(clean(r.get("format")), -1) for r in batch_records], dtype=torch.long)
    texts  = [r["text"] for r in batch_records]
    return texts, labels, meta_v, meta_s, meta_q

model.train()
optimizer.zero_grad()
opt_step, running_loss = 0, 0.0

for micro_step, batch_indices in enumerate(sampler):
    batch_records = [records[i] for i in batch_indices]
    texts, labels, meta_v, meta_s, meta_q = make_tensors(batch_records)

    enc = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
    labels, meta_v, meta_s, meta_q = labels.to(DEVICE), meta_v.to(DEVICE), meta_s.to(DEVICE), meta_q.to(DEVICE)

    with torch.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu",
                         dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
        z = model(enc["input_ids"], enc["attention_mask"])
        loss = ld_supcon_loss(z, labels, meta_v, meta_s, meta_q)

    (loss / ACCUM_STEPS if not (torch.cuda.is_available() and not USE_BF16)
     else scaler.scale(loss / ACCUM_STEPS)).backward()

    running_loss += loss.item()

    if (micro_step + 1) % ACCUM_STEPS == 0:
        if torch.cuda.is_available() and not USE_BF16:
            scaler.step(optimizer); scaler.update()
        else:
            optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        opt_step += 1
        print(f"opt_step={opt_step}/{TOTAL_OPT_STEPS} | avg_loss={running_loss/ACCUM_STEPS:.4f} | lr={scheduler.get_last_lr()[0]:.2e}")
        running_loss = 0.0

    if opt_step >= SMOKE_TEST_MAX_OPT_STEPS:
        break

torch.save(model.state_dict(), "best_pacl_model_smoketest.pt")
print("✅ اسموک‌تست بدون خطا تمام شد؛ چک‌پوینتِ آزمایشی ذخیره شد.")

In [ ]:
import random, math
from collections import defaultdict

class SchoolGroupedPKBatchSampler:
    def __init__(self, records, poet_key="poet", school_key="school",
                 primary_poets_per_batch=3, secondary_poets_per_batch=1,
                 poems_per_poet=2, seed=None):
        self.records = records
        self.P_primary = primary_poets_per_batch
        self.P_secondary = secondary_poets_per_batch
        self.K = poems_per_poet
        self.rng = random.Random(seed)

        self.by_poet = defaultdict(list)
        poet_school = {}
        for idx, r in enumerate(records):
            self.by_poet[r[poet_key]].append(idx)
            poet_school[r[poet_key]] = r[school_key]

        self.school_to_poets = defaultdict(list)
        for poet, school in poet_school.items():
            self.school_to_poets[school].append(poet)

    def _sample_poems(self, poet, k):
        pool = self.by_poet[poet]
        if len(pool) >= k:
            return self.rng.sample(pool, k)
        return [self.rng.choice(pool) for _ in range(k)]

    def __iter__(self):
        queues = {s: self.rng.sample(p, len(p)) for s, p in self.school_to_poets.items()}

        while any(queues.values()):
            primary_school = max(queues, key=lambda s: len(queues[s]))
            if not queues[primary_school]:
                break
            primary_poets = [queues[primary_school].pop()
                              for _ in range(min(self.P_primary, len(queues[primary_school])))]

            other_schools = [s for s in queues if s != primary_school and queues[s]]
            secondary_poets = []
            if other_schools:
                sec_school = self.rng.choice(other_schools)
                secondary_poets = [queues[sec_school].pop()
                                    for _ in range(min(self.P_secondary, len(queues[sec_school])))]

            batch_poets = primary_poets + secondary_poets
            if not batch_poets:
                continue
            batch_indices = []
            for poet in batch_poets:
                batch_indices.extend(self._sample_poems(poet, self.K))
            yield batch_indices

In [ ]:
import os, math, torch

NUM_EPOCHS = 50
ACCUM_STEPS = 32
CKPT_DIR = f"{PHASE3_DIR}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
LAST_CKPT, BEST_CKPT = f"{CKPT_DIR}/last_pacl_model.pt", f"{CKPT_DIR}/best_pacl_model.pt"

def count_batches(seed):
    return sum(1 for _ in SchoolGroupedPKBatchSampler(
        records, primary_poets_per_batch=3, secondary_poets_per_batch=1, poems_per_poet=2, seed=seed))

total_micro_batches = sum(count_batches(42 + e) for e in range(NUM_EPOCHS))
TOTAL_OPT_STEPS = total_micro_batches // ACCUM_STEPS
print(f"کل micro-batch در {NUM_EPOCHS} epoch: {total_micro_batches} → کل opt-step واقعی: {TOTAL_OPT_STEPS}")

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = CosineAnnealingLR(optimizer, T_max=max(TOTAL_OPT_STEPS, 1))

start_epoch, opt_step, best_loss, micro_counter = 0, 0, float("inf"), 0
if os.path.exists(LAST_CKPT):
    ckpt = torch.load(LAST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model"]); optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch, opt_step, best_loss = ckpt["epoch"] + 1, ckpt["opt_step"], ckpt["best_loss"]
    print(f"▶️ ادامه از epoch {start_epoch}")
else:
    print("▶️ شروعِ آموزش از صفر")

model.train()
optimizer.zero_grad()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_sampler = SchoolGroupedPKBatchSampler(
        records, primary_poets_per_batch=3, secondary_poets_per_batch=1, poems_per_poet=2, seed=42 + epoch)
    epoch_loss_sum, epoch_n = 0.0, 0

    for batch_indices in epoch_sampler:
        batch_records = [records[i] for i in batch_indices]
        texts, labels, meta_v, meta_s, meta_q = make_tensors(batch_records)
        enc = tokenizer(texts, padding=True, truncation=True, max_length=256, return_tensors="pt").to(DEVICE)
        labels, meta_v, meta_s, meta_q = labels.to(DEVICE), meta_v.to(DEVICE), meta_s.to(DEVICE), meta_q.to(DEVICE)

        with torch.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu",
                             dtype=AMP_DTYPE, enabled=torch.cuda.is_available()):
            z = model(enc["input_ids"], enc["attention_mask"])
            loss = ld_supcon_loss(z, labels, meta_v, meta_s, meta_q)

        (scaler.scale(loss / ACCUM_STEPS) if (torch.cuda.is_available() and not USE_BF16)
         else (loss / ACCUM_STEPS)).backward()

        epoch_loss_sum += loss.item(); epoch_n += 1
        micro_counter += 1

        if micro_counter % ACCUM_STEPS == 0:
            if torch.cuda.is_available() and not USE_BF16:
                scaler.step(optimizer); scaler.update()
            else:
                optimizer.step()
            scheduler.step(); optimizer.zero_grad(); opt_step += 1

    epoch_avg_loss = epoch_loss_sum / max(epoch_n, 1)
    print(f"epoch={epoch+1}/{NUM_EPOCHS} | batches_this_epoch={epoch_n} | opt_step={opt_step}/{TOTAL_OPT_STEPS} | "
          f"avg_loss={epoch_avg_loss:.4f} | lr={scheduler.get_last_lr()[0]:.2e}")

    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
              "epoch": epoch, "opt_step": opt_step, "best_loss": best_loss, "epoch_avg_loss": epoch_avg_loss}
    torch.save(state, LAST_CKPT)
    if epoch_avg_loss < best_loss:
        best_loss = epoch_avg_loss; state["best_loss"] = best_loss
        torch.save(state, BEST_CKPT)
        print(f"  💾 بهترین مدل ذخیره شد (loss={best_loss:.4f})")

print("✅ تمام شد. بهترین loss:", best_loss)

In [ ]:
import json, math, torch, numpy as np
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5EncoderModel, ByT5Tokenizer
from tqdm.auto import tqdm

PHASE3_DIR = "/content/drive/MyDrive/PACL_Project/phase3_outputs"
MODEL_NAME = "google/byt5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BEST_CKPT = f"{PHASE3_DIR}/checkpoints/best_pacl_model.pt"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

class ByT5PACLEncoder(nn.Module):
    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.encoder = T5EncoderModel.from_pretrained(model_name)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-8)
        return F.normalize(pooled, dim=1)

tokenizer = ByT5Tokenizer.from_pretrained(MODEL_NAME)
model = ByT5PACLEncoder().to(DEVICE)
ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

@torch.no_grad()
def encode_texts(texts, batch_size=64):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        z = model(enc["input_ids"], enc["attention_mask"])
        embs.append(z.float().cpu().numpy())
    return np.concatenate(embs, axis=0)

db_records = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
db_texts = [r["text"] for r in db_records]
db_poets = [r["poet"] for r in db_records]
print(f"تعدادِ رکوردهایِ دیتابیس: {len(db_records)}")

db_embeddings = encode_texts(db_texts, batch_size=64)
print("شکلِ بردارهایِ دیتابیس:", db_embeddings.shape)

n_nan = np.isnan(db_embeddings).any(axis=1).sum()
print(f"تعدادِ ردیف‌هایِ آلوده به NaN: {n_nan} از {len(db_embeddings)}")
assert n_nan == 0, "هنوز NaN وجود دارد — قبل از ادامه باید بررسی شود!"

np.save(f"{PHASE3_DIR}/db_embeddings.npy", db_embeddings)
with open(f"{PHASE3_DIR}/db_poets.json", "w", encoding="utf-8") as f:
    json.dump(db_poets, f, ensure_ascii=False)

print("✅ دیتابیسِ بردارها با موفقیت و بدونِ NaN بازسازی و ذخیره شد.")

In [ ]:
import faiss, torch

torch.cuda.empty_cache()

d = db_embeddings.shape[1]
index_cpu = faiss.IndexFlatIP(d)
index_cpu.add(db_embeddings.astype("float32"))

res = faiss.StandardGpuResources()
res.noTempMemory()
index_gpu = faiss.index_cpu_to_gpu(res, 0, index_cpu)

print(f"ایندکسِ FAISS ساخته شد: {index_gpu.ntotal} بردار، بُعد={d}")
faiss.write_index(index_cpu, f"{PHASE3_DIR}/checkpoints/faiss_db.index")

In [ ]:
test_closed_records = load_jsonl(f"{PHASE3_DIR}/test_closed.jsonl")

In [ ]:
from collections import defaultdict

def identify_poet(raw_text, k=10):
    emb = encode_texts([raw_text], batch_size=1)
    scores, idxs = index_gpu.search(emb.astype("float32"), k)
    scores, idxs = scores[0], idxs[0]

    poet_scores = defaultdict(list)
    for score, idx in zip(scores, idxs):
        if idx == -1:
            continue
        poet_scores[db_poets[idx]].append(float(score))

    ranked = sorted(
        ((poet, max(s_list), len(s_list)) for poet, s_list in poet_scores.items()),
        key=lambda x: x[1], reverse=True
    )
    return [{"poet": poet, "score": score, "votes": votes} for poet, score, votes in ranked]

In [ ]:
sample_text = test_closed_records[0]["text"]
print("شاعرِ واقعی:", test_closed_records[0]["poet"])
for r in identify_poet(sample_text, k=10):
    print(r)

In [ ]:
import random
random.seed(42)
sample = random.sample(test_closed_records, 500)

sample_texts = [r["text"] for r in sample]
sample_true_poets = [r["poet"] for r in sample]
sample_embs = encode_texts(sample_texts, batch_size=64)

K = 20
scores, idxs = index_gpu.search(sample_embs.astype("float32"), K)

def recall_at_k(k):
    hits = sum(
        1 for row_idx, true_poet in enumerate(sample_true_poets)
        if true_poet in [db_poets[i] for i in idxs[row_idx][:k]]
    )
    return hits / len(sample_true_poets)

for k in [1, 5, 10, 20]:
    print(f"Recall@{k}: {recall_at_k(k):.4f}")